In [ ]:
import json
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageDraw
from scipy import stats

## Fastai HRNet result

### analysis val res from fastai hrnet32

In [ ]:
hrnet32 = json.load(open("out_fastai/val_res_hrnet32.json"))

In [ ]:
val_both = json.load(open("annotation_images/trainval_both_17_cp.json"))

In [ ]:
def get_fastai_val_kps(image_id):
    return hrnet32[image_id]

In [ ]:
model_res = {"hrnet32": hrnet32}

hip_regions = ['iliac_crest_left', 'iliac_crest_right', 'iliac_spine_left', 'iliac_spine_right', 'iliopubic_eminence_left', 'iliopubic_eminence_right', 'inferior_pubic_ramus_left', 'inferior_pubic_ramus_right', 'pubic_arch', 'sciatic_notch_left', 'sciatic_notch_right', 'sacrum', 'pubic_tubercle', 'trochanter_left', 'trochanter_right', 'obturator_left', 'obturator_righ']

In [ ]:
def cal_euclidean_distance(x1, y1, x2, y2):
    return np.sqrt((x1-x2)**2 + (y1-y2)**2)

In [ ]:
def get_true_val_kps(image_id):
    for i in range(len(val_both['annotations'])):
        if val_both['annotations'][i]['image_id'] == image_id:
            return np.array(val_both['annotations'][i]['keypoints']).reshape(-1, 3)[:, :-1]

In [ ]:
all_model_df = pd.DataFrame(columns = hip_regions + ['model', 'image_id'])

for model_name, model in model_res.items():
    df = pd.DataFrame(columns = hip_regions)
    for i in range(len(list(hrnet32.keys()))):
        true_kps = get_true_val_kps(int(list(hrnet32.keys())[i]))
        val_kps = get_fastai_val_kps(list(hrnet32.keys())[i])
        for j in range(len(hip_regions)):
            ed = cal_euclidean_distance(true_kps[j][0], true_kps[j][1], val_kps[j][0], val_kps[j][1])
            df.loc[i, hip_regions[j]] = ed
        df.loc[i, 'image_id'] = "{:05d}".format(int(list(hrnet32.keys())[i]))
    df['model'] = model_name
    all_model_df = pd.concat([all_model_df, df])

In [ ]:
all_model_df.head()

In [ ]:
all_model_df.to_csv("./results/pred_vs_anno_val.csv", index = False)

In [ ]:
# test
melt_subset_df = all_model_df.melt(id_vars = 'model', value_vars = hip_regions)
order = melt_subset_df.groupby('variable')['value'].median().sort_values(ascending = False).index
sns.set_style("ticks")
plt.figure(figsize=(5, 7))
sns.boxplot(
    data=melt_subset_df, x="value", y="variable",
    order=order,
    flierprops={"marker": "o"},
    boxprops={"facecolor": (.4, .6, .8, .5)},
)
plt.ylabel("")
plt.xlabel("Euclidean distance (pixel)")
# plt.xlim(0, 7)
plt.title("Euclidean distance between true and predicted hip region")
plt.savefig("./results/edistance_hipregion_cp_fastai.pdf", bbox_inches = 'tight')

#### look at outliers

In [ ]:
hrnet32 = json.load(open("out_fastai/val_res_hrnet32.json"))
val_both = json.load(open("annotation_images/trainval_both_17_cp.json"))
pred_kps = hrnet32["949"][2]
true_kps = np.array([i for i in val_both['annotations'] if i['image_id'] == int("949")][0]['keypoints']).reshape(-1, 3)[:, :-1][2]

img = Image.open(os.path.join("annotation_images/82_93_central_crop/", '{:0>5}.jpg'.format("949")))
rgb_img = Image.new("RGBA", img.size)
rgb_img.paste(img)

draw = ImageDraw.Draw(rgb_img)

print(pred_kps)
print(true_kps)

x, y = pred_kps
draw.ellipse((x-2, y-2, x+2, y+2), fill=('red'))

x, y = true_kps
draw.ellipse((x-2, y-2, x+2, y+2), fill=('green'))

In [ ]:
rgb_img

In [ ]:
hrnet32 = json.load(open("out_fastai/val_res_hrnet32.json"))
val_both = json.load(open("annotation_images/trainval_both_17_cp.json"))
pred_kps = hrnet32["11768"][7]
true_kps = np.array([i for i in val_both['annotations'] if i['image_id'] == int("11768")][0]['keypoints']).reshape(-1, 3)[:, :-1][7]

img = Image.open(os.path.join("annotation_images/82_93_central_crop/", '{:0>5}.jpg'.format("11768")))
rgb_img = Image.new("RGBA", img.size)
rgb_img.paste(img)

draw = ImageDraw.Draw(rgb_img)

print(pred_kps)
print(true_kps)

x, y = pred_kps
draw.ellipse((x-2, y-2, x+2, y+2), fill=('red'))

x, y = true_kps
draw.ellipse((x-2, y-2, x+2, y+2), fill=('green'))

In [ ]:
rgb_img

In [ ]:
hrnet32 = json.load(open("out_fastai/val_res_hrnet32.json"))
val_both = json.load(open("annotation_images/trainval_both_17_cp.json"))
pred_kps = hrnet32["11768"]
true_kps = np.array([i for i in val_both['annotations'] if i['image_id'] == int("11768")][0]['keypoints']).reshape(-1, 3)[:, :-1]

img = Image.open(os.path.join("annotation_images/82_93_central_crop/", '{:0>5}.jpg'.format("11768")))
rgb_img = Image.new("RGBA", img.size)
rgb_img.paste(img)

draw = ImageDraw.Draw(rgb_img)

# print(pred_kps)
# print(true_kps)

for point in pred_kps:
    x, y = point
    draw.ellipse((x-2, y-2, x+2, y+2), fill=('red'))

for point in true_kps:
    x, y = point
    draw.ellipse((x-2, y-2, x+2, y+2), fill=('green'))
    
rgb_img

### Get phenotypes (pixels)

In [ ]:
hip_regions = ['iliac_crest_left', 'iliac_crest_right', 'iliac_spine_left', 'iliac_spine_right', 'iliopubic_eminence_left', 'iliopubic_eminence_right', 'inferior_pubic_ramus_left', 'inferior_pubic_ramus_right', 'pubic_arch', 'sciatic_notch_left', 'sciatic_notch_right', 'sacrum', 'pubic_tubercle', 'trochanter_left', 'trochanter_right', 'obturator_left', 'obturator_right']

In [ ]:
def cal_euclidean_distance(x1, y1, x2, y2):
    return np.sqrt((x1-x2)**2 + (y1-y2)**2)

In [ ]:
hrnet32 = json.load(open("out_fastai/val_res_hrnet32.json"))
val_both = json.load(open("annotation_images/trainval_both_17_cp.json"))

In [ ]:
# get annotation phenotypes

columns = []

for i in range(len(hip_regions)):
    for j in range(i+1, len(hip_regions)):
        columns.append(hip_regions[i] + '2' + hip_regions[j])
        
index = [str(i['image_id']) for i in val_both['annotations']]

val_distance = pd.DataFrame(index = index, columns = columns)

for i in range(len(val_both['annotations'])):
    idx = str(val_both['annotations'][i]['image_id'])
    kps = np.array(val_both['annotations'][i]['keypoints']).reshape(-1, 3)[:, :-1]
    for m in range(len(hip_regions)):
        for n in range(m+1, len(hip_regions)):
            ed = cal_euclidean_distance(kps[m][0],
                                        kps[m][1],
                                        kps[n][0],
                                        kps[n][1])
            val_distance.loc[idx, hip_regions[m] + '2' + hip_regions[n]] = ed

In [ ]:
# get prediction phenotypes

columns = []

for i in range(len(hip_regions)):
    for j in range(i+1, len(hip_regions)):
        columns.append(hip_regions[i] + '2' + hip_regions[j])
        
index = list(hrnet32.keys())

pred_distance = pd.DataFrame(index = index, columns = columns)

for i in range(len(list(hrnet32.keys()))):
    idx = list(hrnet32.keys())[i]
    kps = hrnet32[idx]
    for m in range(len(hip_regions)):
        for n in range(m+1, len(hip_regions)):
            ed = cal_euclidean_distance(kps[m][0],
                                        kps[m][1],
                                        kps[n][0],
                                        kps[n][1])
            pred_distance.loc[idx, hip_regions[m] + '2' + hip_regions[n]] = ed

In [ ]:
pred_distance.head()

In [ ]:
val_distance_filter = val_distance[val_distance.index.isin(pred_distance.index)]

In [ ]:
val_distance_filter.shape

In [ ]:
columns = []

for i in range(len(hip_regions)):
    for j in range(i+1, len(hip_regions)):
        columns.append(hip_regions[i] + '2' + hip_regions[j])
        
index = list(hrnet32.keys())

distance_err = pd.DataFrame(index = index, columns = columns)

for idx in index:
    for col in columns:
        distance_err.loc[idx, col] = np.abs(val_distance_filter.loc[idx, col] -
                                            pred_distance.loc[idx, col])

In [ ]:
distance_err.head()

#### get image_size info

In [ ]:
dtypes = {"Image ID": object}
df_82 = pd.read_csv('./annotation_images/816_288_Patient_EID_master_list_v2.csv', sep = '\t', dtype = dtypes)
df_93 = pd.read_csv('./annotation_images/960_384_Patient_EID_master_list_v2.csv', sep = '\t', dtype = dtypes)
dict_82 = df_82[['File', "Image ID"]].set_index("File")['Image ID'].to_dict()
dict_93 = df_93[['File', "Image ID"]].set_index("File")['Image ID'].to_dict()

In [ ]:
tv_both = json.load(open("annotation_images/trainval_both_17_cp.json"))
tv_82 = json.load(open("annotation_images/trainval_816_288_17_cp.json"))
tv_93 = json.load(open("annotation_images/trainval_960_384_17_cp.json"))

In [ ]:
tv_ids = [i['id'] for i in tv_both['images']]
columns = ['image_id', 'image_size', 'file_name']
tv_mapping = pd.DataFrame(index = tv_ids, columns = columns)

for id in tv_ids:
    img_id = int(id)
    if img_id in [i['image_id'] for i in tv_82['annotations']]:
        image_size = '816x288'
    elif img_id in [i['image_id'] for i in tv_93['annotations']]:
        image_size = '960x384'
    else:
        print("Image not found")
        
    if image_size == '816x288':
        file_name = next(iter({k for k, v in dict_82.items() if int(v) == int(img_id)}))
    elif image_size == '960x384':
        file_name = next(iter({k for k, v in dict_93.items() if int(v) == int(img_id)}))
    
    tv_mapping.loc[img_id, 'image_id'] = img_id
    tv_mapping.loc[img_id, 'image_size'] = image_size
    tv_mapping.loc[img_id, 'file_name'] = file_name

In [ ]:
len(set(tv_mapping['image_id'])) # -> output = 293, which means all image ids are unique, so for one image id, there is only one image size

In [ ]:
tv_mapping[tv_mapping['image_id'] == 15931]['file_name'].values[0]

In [ ]:
tv_mapping

In [ ]:
tv_mapping['file_name'] = tv_mapping['file_name'].apply(lambda x: x[:-4])
tv_mapping.to_csv("annotation_images/tv_mapping.csv", index = False)

In [ ]:
index = distance_err.index
columns = ['image_id', 'image_size', 'file_name']
val_mapping = pd.DataFrame(index = index, columns = columns)

for idx in index:
    img_id = int(idx)
    if img_id in [i['image_id'] for i in tv_82['annotations']]:
        image_size = '816x288'
    elif img_id in [i['image_id'] for i in tv_93['annotations']]:
        image_size = '960x384'
    else:
        print("Image not found")
        
    if image_size == '816x288':
        file_name = next(iter({k for k, v in dict_82.items() if int(v) == int(idx)}))
    elif image_size == '960x384':
        file_name = next(iter({k for k, v in dict_93.items() if int(v) == int(idx)}))
    
    val_mapping.loc[idx, 'image_id'] = img_id
    val_mapping.loc[idx, 'image_size'] = image_size
    val_mapping.loc[idx, 'file_name'] = file_name

In [ ]:
val_mapping['file_name'] = val_mapping['file_name'].apply(lambda x: x[:-4])

In [ ]:
val_mapping.to_csv("annotation_images/val_mapping.csv", index = False)

#### convert pixels to cm

In [ ]:
val_mapping = pd.read_csv("annotation_images/val_mapping.csv"); val_mapping.head()

In [ ]:
p_all_info_z_filtered = pd.read_csv("all_prediction/p_all_info_z_filtered.csv")
p_all_info_z_filtered = p_all_info_z_filtered[['file_name', 'ratio']]; p_all_info_z_filtered.head()

In [ ]:
val_mapping['image_id'] = val_mapping['image_id'].apply(lambda x: str(x))

In [ ]:
pred_distance = pred_distance.merge(val_mapping, left_index = True, right_on = 'image_id', how = 'inner')
pred_distance = pred_distance.merge(p_all_info_z_filtered, left_on = 'file_name', right_on = 'file_name', how = 'inner')

for i in range(len(hip_regions)):
    for j in range(i+1, len(hip_regions)):
        pred_distance[hip_regions[i] + '2' + hip_regions[j]] = pred_distance[hip_regions[i] + '2' + hip_regions[j]] * pred_distance['ratio']

In [ ]:
pred_distance.head()

In [ ]:
val_distance = val_distance.merge(val_mapping, left_index = True, right_on = 'image_id', how = 'inner')
val_distance = val_distance.merge(p_all_info_z_filtered, left_on = 'file_name', right_on = 'file_name', how = 'inner')

for i in range(len(hip_regions)):
    for j in range(i+1, len(hip_regions)):
        val_distance[hip_regions[i] + '2' + hip_regions[j]] = val_distance[hip_regions[i] + '2' + hip_regions[j]] * val_distance['ratio']

In [ ]:
val_distance.head()

In [ ]:
set(pred_distance['image_id']) == set(val_distance['image_id'])

### Use prediction as training set

Visually speaking, the model predictions looks better than human manually annotation, so we need to provide the evidence to support this claim. So we use prediction result as training set annotation and train the model again

In [ ]:
# load data

# prediction result -- using prediction as train
val_res_p_as_t_same = json.load(open("all_prediction/val_res_pred_as_train_hrnet32_same.json"))
val_res_p_as_t = json.load(open("all_prediction/val_res_pred_as_train_hrnet32.json"))

# prediction result -- using human annotation as train
hrnet32_82_pred = json.load(open("all_prediction/hrnet32_82_pred.json"))
hrnet32_93_pred = json.load(open("all_prediction/hrnet32_93_pred.json"))

In [ ]:
def cal_euclidean_distance(x1, y1, x2, y2):
    return np.sqrt((x1-x2)**2 + (y1-y2)**2)

#### Use prediction as training set - same images as human annotation

In [ ]:
[i for i in list(val_res_p_as_t_same.keys()) if i.split("_")[0] not in list(hrnet32_82_pred.keys()) and i.split("_")[0] not in list(hrnet32_93_pred.keys())]

In [ ]:
hip_regions = ['iliac_crest_left', 'iliac_crest_right', 'iliac_spine_left', 'iliac_spine_right', 'iliopubic_eminence_left', 'iliopubic_eminence_right', 'inferior_pubic_ramus_left', 'inferior_pubic_ramus_right', 'pubic_arch', 'sciatic_notch_left', 'sciatic_notch_right', 'sacrum', 'pubic_tubercle', 'trochanter_left', 'trochanter_right', 'obturator_left', 'obturator_righ']

In [ ]:
val_name = sorted(list(val_res_p_as_t_same.keys()))

In [ ]:
for i in val_name:
    print(i)

In [ ]:
val_name = sorted(list(val_res_p_as_t_same.keys()))

error_df = pd.DataFrame(columns = hip_regions, index = val_name)

for image_name in val_name:
    img_id = image_name.split("_")[0]
    img_size = image_name.split("_")[1]
    if img_size == '82':
        pred_kps = hrnet32_82_pred[img_id]
    elif img_size == '93':
        pred_kps = hrnet32_93_pred[img_id]
    ppred_kps = val_res_p_as_t_same[image_name]
    for i in range(len(hip_regions)):
        error_df.loc[image_name, hip_regions[i]] = cal_euclidean_distance(pred_kps[i][0], 
                                                                          pred_kps[i][1], 
                                                                          ppred_kps[i][0], 
                                                                          ppred_kps[i][1])

In [ ]:
error_df.head()

In [ ]:
error_df.to_csv("./results/ppred_vs_pred_val_same_img.csv", index = True)

In [ ]:
error_df.melt()

In [ ]:
# plot error
melt_error_df = error_df.melt()
order = melt_error_df.groupby('variable')['value'].median().sort_values(ascending = False).index
sns.set_style("ticks")
plt.figure(figsize=(5, 7))
sns.boxplot(
    data=melt_error_df, x="value", y="variable",
    order=order,
    flierprops={"marker": "o"},
    boxprops={"facecolor": (.4, .6, .8, .5)},
)
plt.ylabel("")
plt.xlabel("Euclidean distance (pixel)")
# plt.xlim(0, 7)
plt.title("Euclidean distance between true and predicted hip region")
plt.savefig("./results/ppred_vs_pred_val_same_img_ed.pdf", bbox_inches = 'tight')

#### Use prediction as training set - random sampled images

In [ ]:
[i for i in list(val_res_p_as_t.keys()) if i.split("_")[0] not in list(hrnet32_82_pred.keys()) and i.split("_")[0] not in list(hrnet32_93_pred.keys())]

In [ ]:
hip_regions = ['iliac_crest_left', 'iliac_crest_right', 'iliac_spine_left', 'iliac_spine_right', 'iliopubic_eminence_left', 'iliopubic_eminence_right', 'inferior_pubic_ramus_left', 'inferior_pubic_ramus_right', 'pubic_arch', 'sciatic_notch_left', 'sciatic_notch_right', 'sacrum', 'pubic_tubercle', 'trochanter_left', 'trochanter_right', 'obturator_left', 'obturator_righ']

In [ ]:
val_name = sorted(list(val_res_p_as_t.keys()))

In [ ]:
for i in val_name:
    print(i)

In [ ]:
val_name = sorted(list(val_res_p_as_t.keys()))

error_df = pd.DataFrame(columns = hip_regions, index = val_name)

for image_name in val_name:
    img_id = image_name.split("_")[0]
    img_size = image_name.split("_")[1]
    if img_size == '82':
        pred_kps = hrnet32_82_pred[img_id]
    elif img_size == '93':
        pred_kps = hrnet32_93_pred[img_id]
    ppred_kps = val_res_p_as_t[image_name]
    for i in range(len(hip_regions)):
        error_df.loc[image_name, hip_regions[i]] = cal_euclidean_distance(pred_kps[i][0], 
                                                                          pred_kps[i][1], 
                                                                          ppred_kps[i][0], 
                                                                          ppred_kps[i][1])

In [ ]:
error_df.head()

In [ ]:
error_df.to_csv("./results/ppred_vs_pred_val.csv", index = True)

In [ ]:
error_df.melt()

In [ ]:
# plot error
melt_error_df = error_df.melt()
order = melt_error_df.groupby('variable')['value'].median().sort_values(ascending = False).index
sns.set_style("ticks")
plt.figure(figsize=(5, 7))
sns.boxplot(
    data=melt_error_df, x="value", y="variable",
    order=order,
    flierprops={"marker": "o"},
    boxprops={"facecolor": (.4, .6, .8, .5)},
)
plt.ylabel("")
plt.xlabel("Euclidean distance (pixel)")
# plt.xlim(0, 7)
plt.title("Euclidean distance between true and predicted hip region")
plt.savefig("./results/ppred_vs_pred_val_ed.pdf", bbox_inches = 'tight')

### Training log

In [ ]:
pwd

In [ ]:
panno = pd.read_clipboard()
panno.to_csv('key_results/pred_on_anno_training_log.csv', index=False)

In [ ]:
ppred = pd.read_clipboard()
ppred.to_csv('key_results/pred_on_pred_training_log.csv', index=False)

In [ ]:
panno['Type'] = 'predict on manual annotation'
ppred['Type'] = 'predict on 1st round prediction result'

# concat
pall = pd.concat([panno, ppred], axis=0)
pall_melt = pall.melt(id_vars=['epoch', 'nmae_topk', 'Type'], value_vars=['train_loss', 'valid_loss'], var_name='Dataset', value_name='Loss')

In [ ]:
sns.lineplot(x="epoch", y="Loss", data=pall_melt, hue='Type', style = 'Dataset')
plt.xlabel('Epoch', fontname="Helvetica", fontsize=12)
plt.ylabel('Loss', fontname="Helvetica", fontsize=12)
plt.legend(frameon=False)

# Setting font for the tick labels
plt.xticks(fontname="Helvetica", fontsize=10)
plt.yticks(fontname="Helvetica", fontsize=10)

# Setting font for legend
legend = plt.legend(frameon=False)
for text in legend.get_texts():
    text.set_fontname('Helvetica')
    text.set_fontsize(12)

plt.savefig('out_fig/training_log.pdf', bbox_inches='tight')

### compare to human annotation

In [ ]:
hip_regions = ['iliac_crest_left', 'iliac_crest_right', 'iliac_spine_left', 'iliac_spine_right', 'iliopubic_eminence_left', 'iliopubic_eminence_right', 'inferior_pubic_ramus_left', 'inferior_pubic_ramus_right', 'pubic_arch', 'sciatic_notch_left', 'sciatic_notch_right', 'sacrum', 'pubic_tubercle', 'trochanter_left', 'trochanter_right', 'obturator_left', 'obturator_righ']

In [ ]:
ppred = pd.read_csv("./results/ppred_vs_pred_val.csv", index_col=0); ppred.head()

In [ ]:
ppred_same = pd.read_csv("./results/ppred_vs_pred_val_same_img.csv", index_col=0); ppred_same.head()

In [ ]:
h_anno = pd.read_csv("./results/pred_vs_anno_val.csv")[hip_regions]; h_anno.head()

In [ ]:
ppred_melt = ppred.melt()
ppred_melt['source'] = 'pred_on_pred'

In [ ]:
ppred_melt.head()

In [ ]:
ppred_same_melt = ppred_same.melt()
ppred_same_melt['source'] = 'pred_on_pred_same_img'

In [ ]:
ppred_same_melt.head()

In [ ]:
h_anno_melt = h_anno.melt()
h_anno_melt['source'] = 'pred_on_anno'

In [ ]:
h_anno_melt.head()

In [ ]:
all_melt = pd.concat([ppred_melt, ppred_same_melt, h_anno_melt], axis = 0); all_melt.head()

In [ ]:
# overall error comparison
sns.set_theme(style="ticks")
# p = stats.ttest_ind(ppred_melt['value'].tolist(), h_anno_melt['value'].tolist())[1]
sns.set_style("ticks")
plt.figure(figsize=(5, 5))
sns.boxplot(x = "value", y = "source", data = all_melt, 
            flierprops={"marker": "o"})
plt.xlabel("Euclidean distance (pixel)", size = 15)
plt.ylabel("")
# plt.ylim(-1, 11)
plt.xticks(size = 15)
plt.yticks(size = 15)
# plt.text(x = 0.1, y = 10, s = "p-value = {:.3e}".format(p), fontsize = 15)
plt.savefig("./results/ppred_vs_pred.pdf", bbox_inches = 'tight')

In [ ]:
plt.figure(figsize=(5, 15))
sns.boxplot(x="value", y="variable",
            hue="source",
            flierprops={"marker": "o", "markerfacecolor": "black", "markersize": 3},
            data=all_melt)
sns.despine(offset=10, trim=True)
plt.ylabel("")
plt.xlabel("Euclidean distance (pixel)")
plt.savefig("./results/ppred_vs_pred_split_ed.pdf", bbox_inches = 'tight')

### prediction vs human annotation (2023-07-16)

In [ ]:
# copy paste from tacc jupyter notebook

val_set = """[Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/11276_93.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/06157_93.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/08221_93.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/01460_82.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/13299_82.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/00157_82.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/15233_82.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/12106_82.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/11703_82.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/10377_82.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/09968_82.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/10466_93.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/20068_82.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/01061_82.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/14914_93.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/00631_82.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/18529_82.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/07237_93.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/11768_93.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/01643_82.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/02957_82.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/03145_93.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/16212_82.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/21017_82.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/20181_82.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/03368_93.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/09869_82.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/14445_93.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/04809_93.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/14326_82.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/18085_82.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/07398_82.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/15653_82.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/00093_82.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/13666_93.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/14654_82.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/04549_93.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/01698_93.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/10770_82.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/06770_93.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/07887_82.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/01809_93.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/06938_93.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/01432_93.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/04054_93.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/16295_93.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/11380_82.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/12084_82.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/08760_93.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/02406_82.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/03268_93.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/01729_93.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/14442_82.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/12443_93.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/15803_82.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/05565_93.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/13290_93.jpg'),
 Path('/work2/09059/xliaoyi/frontera/narasimhan_lab/hip_shape/images/tv_images_cp/11910_82.jpg')]"""

In [ ]:
import re

# Regular expression pattern
pattern = r'(\d{5}_\d{2})'

# Find all matches
val_set = re.findall(pattern, val_set)

# Now 'matches' is a list of all "11276_93" like patterns
print(val_set)

In [ ]:
len(val_set)

#### Coordinates for annotation

In [ ]:
# generate annotation keypoints
anno_path = "annotation_images/annotations/"

# mapping dict
dtypes = {"Image ID": object}
df_82 = pd.read_csv(anno_path + '816_288_Patient_EID_master_list_v2.csv', sep = '\t', dtype = dtypes)
df_93 = pd.read_csv(anno_path + '960_384_Patient_EID_master_list_v2.csv', sep = '\t', dtype = dtypes)
dict_82 = df_82[['File', "Image ID"]].set_index("File")['Image ID'].to_dict()
dict_93 = df_93[['File', "Image ID"]].set_index("File")['Image ID'].to_dict()

# load coco annotation result
anno_82 = json.load(open(os.path.join(anno_path, "tv_82_23kps.json")))
anno_93 = json.load(open(os.path.join(anno_path, "tv_93_23kps.json")))

# convert coco result to keypoints
anno_23kps = {}

for img in anno_82['images']:
    img_id = img['id']
    file_name = img['file_name']
    image_id = dict_82[file_name]
    for i in anno_82['annotations']:
        if i['image_id'] == img_id:
            kps = np.array(i['keypoints']).reshape(-1, 3)[:, :2]
            # according to central crop, adjust x,y
            kps[:, 0] = kps[:, 0] - 16
            kps[:, 1] = kps[:, 1] - 230
            kps = kps.tolist()
    anno_23kps[image_id  + '_82'] = kps

for img in anno_93['images']:
    img_id = img['id']
    file_name = img['file_name']
    image_id = dict_93[file_name]
    for i in anno_93['annotations']:
        if i['image_id'] == img_id:
            kps = np.array(i['keypoints']).reshape(-1, 3)[:, :2]
            # according to central crop, adjust x,y
            kps[:, 0] = kps[:, 0] - 64
            kps[:, 1] = kps[:, 1] - 302
            kps = kps.tolist()
    anno_23kps[image_id  + '_93'] = kps

In [ ]:
len(anno_23kps['00093_82'])

In [ ]:
len(anno_23kps.keys())

#### Coordinates for 1st and 2nd prediction

In [ ]:
panno = json.load(open("all_prediction/all_res_pred_on_anno_23.json"))
ppred = json.load(open("all_prediction/all_res_pred_on_pred_23.json"))

In [ ]:
len(panno['15666_82'])

#### Calculate euclidean distance

In [ ]:
def cal_euclidean_distance(x1, y1, x2, y2):
    return np.sqrt((x1-x2)**2 + (y1-y2)**2)

In [ ]:
# hip_regions = ['iliac_crest_left', 'iliac_crest_right', 'iliac_spine_left', 'iliac_spine_right', 'iliopubic_eminence_left', 'iliopubic_eminence_right', 
#                'inferior_pubic_ramus_left', 'inferior_pubic_ramus_right', 'pubic_arch', 'sciatic_notch_left', 'sciatic_notch_right', 'sacrum', 
#                'pubic_tubercle', 'trochanter_left', 'trochanter_right', 'obturator_left', 'obturator_right', 
#                'sacrum_left', 'sacrum_right', 'inferior_iliac_spine_left', 'inferior_iliac_spine_right','acetabular_inferior_left', 'acetabular_inferior_right']

# select hip regions
hip_regions = ['iliac_crest_left', 'iliac_crest_right', 'iliac_spine_left', 'iliac_spine_right', 'iliopubic_eminence_left', 'iliopubic_eminence_right',
         'inferior_pubic_ramus_left', 'inferior_pubic_ramus_right', 'pubic_arch', 'sciatic_notch_left', 'sciatic_notch_right', 'sacrum',
         'pubic_tubercle', 'inferior_iliac_spine_left', 'inferior_iliac_spine_right', 'acetabular_inferior_left', 'acetabular_inferior_right']

err_anno_vs_pred = pd.DataFrame(index = val_set, columns = hip_regions)

for key in val_set:
    for i in range(len(hip_regions)):
        err_anno_vs_pred.loc[key, hip_regions[i]] = cal_euclidean_distance(panno[key][i][0], panno[key][i][1], anno_23kps[key][i][0], anno_23kps[key][i][1])

err_anno_vs_pred = err_anno_vs_pred.reset_index()
err_anno_vs_pred = err_anno_vs_pred.melt(id_vars = 'index', var_name = 'hip_region', value_name = 'error')
err_anno_vs_pred['group'] = 'anno_vs_pred'

err_pred_vs_pred = pd.DataFrame(index = val_set, columns = hip_regions)

for key in val_set:
    for i in range(len(hip_regions)):
        err_pred_vs_pred.loc[key, hip_regions[i]] = cal_euclidean_distance(ppred[key][i][0], ppred[key][i][1], panno[key][i][0], panno[key][i][1])


err_pred_vs_pred = err_pred_vs_pred.reset_index()
err_pred_vs_pred = err_pred_vs_pred.melt(id_vars = 'index', var_name = 'hip_region', value_name = 'error')
err_pred_vs_pred['group'] = 'pred_vs_pred'

err = pd.concat([err_anno_vs_pred, err_pred_vs_pred], axis = 0)

# Mapping from old names to new names
name_mapping = {
    'iliac_crest_left': 'Iliac crest posterior right', 
    'iliac_crest_right': 'Iliac crest posterior left', 
    'iliac_spine_left': 'Iliac crest anterolateral right',
    'iliac_spine_right': 'Iliac crest anterolateral left',
    'iliopubic_eminence_left': 'Acetabulum posterosuperior right',
    'iliopubic_eminence_right': 'Acetabulum posterosuperior left', 
    'inferior_pubic_ramus_left': 'Ischiopubic ramus inferior right',
    'inferior_pubic_ramus_right': 'Ischiopubic ramus inferior left', 
    'pubic_arch': 'Pubic symphysis inferior', 
    'sciatic_notch_left': 'Pelvic inlet right',
    'sciatic_notch_right': 'Pelvic inlet left', 
    'sacrum': 'Sacrum midline', 
    'pubic_tubercle': 'Pubic symphysis superior', 
    'inferior_iliac_spine_left': 'Iliac body lateral right',
    'inferior_iliac_spine_right': 'Iliac body lateral left', 
    'acetabular_inferior_left': 'Acetabulum anteroinferior right',
    'acetabular_inferior_right': 'Acetabulum anteroinferior left'
}

# Apply the mapping to the 'hip_region' column
err['hip_region'] = err['hip_region'].replace(name_mapping)
err.to_csv("key_results/error_23kps_anno_vs_pred_longer.csv", index = False)

err = err_anno_vs_pred.merge(err_pred_vs_pred, on = ['index', 'hip_region'], suffixes = ('_anno_vs_pred', '_pred_vs_pred'))
err['hip_region'] = err['hip_region'].replace(name_mapping)
err.to_csv("key_results/error_23kps_anno_vs_pred_wider.csv", index = False)

In [ ]:
err

In [ ]:
plt.figure(figsize=(5, 5))
sns.boxplot(x="error", y="group",
            flierprops={"marker": "o", "markerfacecolor": "black", "markersize": 3},
            data=err)
sns.despine(offset=10, trim=True)
plt.ylabel("")
plt.xlabel("Euclidean distance (pixel)")